# 05 — Robustness

Three questions, in order of how much damage a bad answer would do:

1. Does the pooled decline survive the **elevation weight**? `C` and `S` share the elevation coordinate, so a reviewer can argue the contraction is built in. Setting `w_z = 0` removes the shared coordinate entirely.
2. Does it survive the **spatial block size**? The bootstrap chops the map into blocks of ~8 cells, a hard-coded choice that sets every confidence interval in the analysis.
3. Are the **districts** distinguishable under any of these settings? (Notebook 02 says no; this confirms it does not depend on tuning.)

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, json
import pipeline as P

rng = np.random.default_rng(7)
isl = P.load_island(); FT, FTemp, reg, xy = P.load_farms(isl)
NB = 400   # replicates per setting; the validated ML scripts use 2,000

def boot_idx(blocks):
    return np.concatenate([np.concatenate([gi[lb == k] for k in rng.integers(0, n, n)])
                           for gi, lb, n in (blocks[r] for r in ('kona','kau'))])

def ci(a):
    a = np.asarray(a); return np.median(a), np.percentile(a,2.5), np.percentile(a,97.5)

## 1. Elevation weight

If the decline vanished at `w_z = 0` the shared-coordinate objection would be fatal.

In [2]:
blocks = P.spatial_blocks(reg, xy, 8)
rows = []
for wz in (0.0, 0.5, 1.0, 1.5, 2.0):
    n0, _ = P.pooled_union(isl, FT, FTemp, reg, wz=wz, dt=0.0)
    n45, _ = P.pooled_union(isl, FT, FTemp, reg, wz=wz, dt=1.35)
    pt = 100*(n45-n0)/n0
    rep = []
    for _ in range(NB):
        i = boot_idx(blocks)
        if (reg[i]=='kau').sum() < 5 or (reg[i]=='kona').sum() < 5: continue
        try:
            a,_ = P.pooled_union(isl, FT, FTemp, reg, idx=i, wz=wz, dt=0.0)
            b,_ = P.pooled_union(isl, FT, FTemp, reg, idx=i, wz=wz, dt=1.35)
            if a > 10: rep.append(100*(b-a)/a)
        except Exception: pass
    _, lo, hi = ci(rep)
    rows.append([wz, pt, lo, hi, 'yes' if hi < 0 else 'NO'])
print(pd.DataFrame(rows, columns=['w_z','decline 2045 %','ci lo','ci hi','excludes 0'])
        .round(1).to_string(index=False))
print('\nw_z = 0 drops elevation from the screen entirely. The decline persists there,')
print('so the shared-coordinate objection cannot account for it. Magnitude is')
print('weight-dependent and should be quoted as a range, not a point.')

 w_z  decline 2045 %  ci lo  ci hi excludes 0
 0.0           -12.0  -20.1   -5.3        yes
 0.5           -14.2  -24.4   -7.9        yes
 1.0           -22.1  -33.8  -12.6        yes
 1.5           -29.1  -44.4  -23.7        yes
 2.0           -37.5  -51.3  -30.4        yes

w_z = 0 drops elevation from the screen entirely. The decline persists there,
so the shared-coordinate objection cannot account for it. Magnitude is
weight-dependent and should be quoted as a range, not a point.


## 2. Block size

`nb = round(n / cells_per_block)` with `cells_per_block = 8` was never checked against the spatial correlation range. It sets every interval width in the analysis.

In [3]:
bs_rows = []
for cpb in (4, 8, 16, 24):
    bl = P.spatial_blocks(reg, xy, cpb)
    rep = []
    for _ in range(NB):
        i = boot_idx(bl)
        if (reg[i]=='kau').sum() < 5 or (reg[i]=='kona').sum() < 5: continue
        try:
            a,_ = P.pooled_union(isl, FT, FTemp, reg, idx=i, dt=0.0)
            b,_ = P.pooled_union(isl, FT, FTemp, reg, idx=i, dt=1.35)
            if a > 10: rep.append(100*(b-a)/a)
        except Exception: pass
    _, lo, hi = ci(rep)
    bs_rows.append([cpb, bl['kona'][2], bl['kau'][2], lo, hi, 'yes' if hi < 0 else 'NO'])
print(pd.DataFrame(bs_rows, columns=['cells/block','kona blocks','kau blocks',
                                  'ci lo','ci hi','excludes 0']).round(1).to_string(index=False))
print('\nThe verdict does not depend on the block size, so no variogram is needed.')

 cells/block  kona blocks  kau blocks  ci lo  ci hi excludes 0
           4          102          16  -28.6  -12.6        yes
           8           51           8  -33.6  -13.2        yes
          16           26           4  -37.2  -12.8        yes
          24           17           4  -38.0  -12.7        yes

The verdict does not depend on the block size, so no variogram is needed.


## 3. District contrast — still null

The same bootstrap, applied to the difference between districts rather than the union.

In [4]:
def district_declines(i):
    s = P.screen(FT, reg, isl['X'], i)
    out = {}
    for r, like in (('kona', s['kona_like']), ('kau', s['kau_like'])):
        mu, sg = P.envelope(FTemp, reg, i, r)
        mask = isl['farmable'] & like
        f0 = P.feasible_size(isl['T'], mask, mu, sg, 0.0)
        if f0 < 10: return None
        f1 = P.feasible_size(isl['T'], mask, mu, sg, 1.35)
        out[r] = 100*(f1-f0)/f0
    return out

base = district_declines(np.arange(len(FT)))
rep = []
for _ in range(NB):
    i = boot_idx(blocks)
    if (reg[i]=='kau').sum() < 5 or (reg[i]=='kona').sum() < 5: continue
    try:
        d = district_declines(i)
        if d: rep.append(d['kona'] - d['kau'])
    except Exception: pass
m, lo, hi = ci(rep)
print(f"kona {base['kona']:+.1f}%   kau {base['kau']:+.1f}%")
print(f"contrast (kona - kau) {base['kona']-base['kau']:+.1f} pp   95% CI ({lo:+.1f}, {hi:+.1f})")
print(f"-> {'DISTINGUISHABLE' if (lo>0 or hi<0) else 'not distinguishable'}")
print('\nConsistent with notebook 02: the districts cannot be separated on trajectory,')
print('which is the empirical basis for reporting the pooled union instead.')

kona -26.4%   kau -18.2%
contrast (kona - kau) -8.2 pp   95% CI (-29.2, +12.3)
-> not distinguishable

Consistent with notebook 02: the districts cannot be separated on trajectory,
which is the empirical basis for reporting the pooled union instead.


## 4. Headroom and dz/h

`ANALYSIS_TODO.md` item 3b. Section 4.4's inversion condition, `dz/h > 1`, has so far been reported as a point estimate: Kona 1.19, Ka'u 1.05. `h` is measured at the upper fringe of the belt, exactly where the screen cross-validates worst, and Ka'u's ratio is close enough to one that the condition should not be asserted for that district without an interval. Same block bootstrap as sections 1-3; refits the screen and the lapse-rate fit inside every replicate.

In [5]:
z = isl['X'][:, 0]   # elevation is the screen's first feature

def headroom(i):
    s = P.screen(FT, reg, isl['X'], i)
    belt = isl['farmable'] & (s['kona_like'] | s['kau_like']) & ~np.isnan(isl['T'])
    if belt.sum() < 10: return None
    fit = np.polyfit(z[belt], isl['T'][belt], 1)
    gamma = abs(fit[0]); dz = P.DT_HORIZON['2045'] / gamma
    out = {'gamma': gamma, 'dz': dz}
    for r, like in (('kona', s['kona_like']), ('kau', s['kau_like'])):
        mu, sg = P.envelope(FTemp, reg, i, r)
        zS = z[isl['farmable'] & like]
        if zS.size < 10: return None
        zmax = np.percentile(zS, 97.5)
        zlead0 = (mu - P.HW*sg - fit[1]) / fit[0]
        h = zmax - zlead0
        out[f'h_{r}'] = h; out[f'ratio_{r}'] = dz / h
    return out

base_h = headroom(np.arange(len(FT)))
keys = ('gamma', 'dz', 'h_kona', 'h_kau', 'ratio_kona', 'ratio_kau')
rep_h = {k: [] for k in keys}
for _ in range(NB):
    i = boot_idx(blocks)
    if (reg[i]=='kau').sum() < 5 or (reg[i]=='kona').sum() < 5: continue
    try:
        out = headroom(i)
    except Exception:
        out = None
    if out:
        for k in keys: rep_h[k].append(out[k])

OUT_H = {}
print(f"{'quantity':>12} {'point':>8} {'95% CI':>20} {'n':>6}")
for k in ('h_kona', 'h_kau', 'ratio_kona', 'ratio_kau'):
    m, lo, hi = ci(rep_h[k])
    OUT_H[f'{k}_boot_median'] = float(m); OUT_H[f'{k}_ci_lo'] = float(lo); OUT_H[f'{k}_ci_hi'] = float(hi)
    print(f"{k:>12} {base_h[k]:>8.2f} {f'({lo:.2f}, {hi:.2f})':>20} {len(rep_h[k]):>6}")

for r in ('kona', 'kau'):
    lo, hi = OUT_H[f'ratio_{r}_ci_lo'], OUT_H[f'ratio_{r}_ci_hi']
    verdict = 'excludes 1 -- inversion condition robust' if lo > 1 else 'includes 1 -- NOT confidently established'
    print(f"\n{r}: dz/h = {base_h[f'ratio_{r}']:.2f}  95% CI ({lo:.2f}, {hi:.2f})  -> {verdict}")

json.dump({**{k: float(base_h[k]) for k in keys}, **OUT_H},
          open('data/headroom_bootstrap.json', 'w'), indent=1)
print('\nwrote data/headroom_bootstrap.json')

    quantity    point               95% CI      n
      h_kona   197.41      (95.27, 283.02)    400
       h_kau   224.00     (113.46, 352.63)    400
  ratio_kona     1.19         (0.83, 2.47)    400
   ratio_kau     1.05         (0.67, 2.08)    400

kona: dz/h = 1.19  95% CI (0.83, 2.47)  -> includes 1 -- NOT confidently established

kau: dz/h = 1.05  95% CI (0.67, 2.08)  -> includes 1 -- NOT confidently established

wrote data/headroom_bootstrap.json


## Figure — the decline survives every discretionary choice

Left: the elevation weight. `w_z = 0` removes elevation from the screen entirely, which is the strongest form of the shared-coordinate objection — the decline persists there. Right: the spatial block size, which sets every interval width in the analysis. Neither changes the verdict; only the magnitude moves.

In [6]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, os
os.makedirs('figures', exist_ok=True)
F_COL='#c1440e'
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})

wz_rows = rows  # from the elevation-weight cell
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
ax = axes[0]
w  = [r[0] for r in wz_rows]; pt = [r[1] for r in wz_rows]
lo = [r[2] for r in wz_rows]; hi = [r[3] for r in wz_rows]
ax.fill_between(w, lo, hi, color=F_COL, alpha=.18, label='95% CI')
ax.plot(w, pt, 'o-', color=F_COL, lw=2, ms=5, label='point estimate')
ax.axhline(0, color='k', lw=1)
# ax.annotate('elevation dropped\nfrom the screen', xy=(0, pt[0]), xytext=(0.12, -46),
#             fontsize=8, arrowprops=dict(arrowstyle='->', lw=.8))
ax.set_xlabel('elevation weight $w_z$'); ax.set_ylabel('|F| change to 2045 (%)')
ax.set_title('Never crosses zero'); ax.legend(frameon=False, fontsize=9)

ax = axes[1]
cp = [r[0] for r in bs_rows]; blo = [r[3] for r in bs_rows]; bhi = [r[4] for r in bs_rows]
ax.errorbar(range(len(cp)), [(a+b)/2 for a,b in zip(blo,bhi)],
            yerr=[[(a+b)/2-a for a,b in zip(blo,bhi)],[b-(a+b)/2 for a,b in zip(blo,bhi)]],
            fmt='o', color=F_COL, capsize=4, lw=2, ms=5)
ax.axhline(0, color='k', lw=1)
ax.set_xticks(range(len(cp)))
ax.set_xticklabels([f'{c}\n({r[1]}/{r[2]} blocks)' for c,r in zip(cp,bs_rows)], fontsize=8)
ax.set_xlabel('cells per spatial block'); ax.set_ylabel('|F| change to 2045 (%)')
ax.set_title('Interval width is insensitive to block size')
fig.tight_layout(); fig.savefig('figures/05_robustness.png', dpi=200, bbox_inches='tight')
plt.close(fig); print('figures/05_robustness.png')

figures/05_robustness.png
